# Clase 1 — Práctica de LLMs: tokenización, embeddings y atención

**Duración estimada:** 2 horas (5 secciones de ~20-25 min cada una).

**Requisitos previos:** Python, algo de ML (haber entrenado/evaluado algún modelo alguna vez).

**Entorno:** este notebook corre tanto en **Google Colab** (recomendado, no requiere GPU) como en
un entorno **local** con Jupyter, siempre que se instalen las librerías de la primera celda.

## Contenido

1. Tokenización comparada (BPE vs WordPiece vs multilingüe)
2. Embeddings clásicos con Word2Vec
3. Visualización de embeddings (t-SNE vs UMAP)
4. Self-Attention y Multi-Head Attention desde cero (PyTorch)
5. Atención en un modelo real (DistilBERT vs GPT-2)

Cada sección tiene: una explicación breve, código comentado, y un **Ejercicio** al final para
completar o modificar algo. No hace falta terminar todos los ejercicios en clase — lo importante
es entender la mecánica de cada celda.


## Instalación de dependencias

Corré esta celda una sola vez por sesión (en Colab, después de cada reinicio de entorno).

In [ ]:
# En Colab, `torch`, `pandas` y `matplotlib` ya suelen venir preinstalados, asi que pip
# los va a saltear si la version instalada ya sirve (no usamos --upgrade para no forzar
# una reinstalacion de torch que podria romper la compatibilidad con la GPU de Colab).
# pip-system-certs no hace nada en Colab, pero en redes corporativas con proxy que
# intercepta HTTPS (certificado propio) evita errores SSLCertVerificationError al
# descargar modelos de Hugging Face: hace que las verificaciones usen el almacen de
# certificados del sistema operativo en vez del bundle de certifi.
#!pip install -q transformers tokenizers gensim scikit-learn umap-learn bertviz torch pandas matplotlib pip-system-certs


## Imports generales

Algunas librerías (`umap-learn`, `bertviz`) pueden fallar en ciertos entornos (por ejemplo,
restricciones de red o de renderizado de HTML/JS). Las importamos con manejo de errores para
que el resto del notebook siga funcionando aunque alguna no esté disponible.

In [ ]:
from __future__ import annotations  # permite usar sintaxis moderna de type hints (X | None, tuple[...])
                                       # incluso en versiones de Python un poco mas viejas

import warnings
warnings.filterwarnings("ignore")  # oculta warnings de librerias (no afecta la ejecucion)

# --- Librerias basicas: arrays, tablas y graficos ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- PyTorch: lo usamos para implementar attention desde cero en la Seccion 4 ---
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Optional

# --- Hugging Face: tokenizers y modelos pre-entrenados (Secciones 1 y 5) ---
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

# --- Reduccion de dimensionalidad para visualizar embeddings (Seccion 3) ---
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# --- Word2Vec y vectores pre-entrenados GloVe (Seccion 2) ---
import gensim
from gensim.models import Word2Vec
import gensim.downloader as gensim_api

# --- Librerias opcionales ---
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("[aviso] umap-learn no disponible: en la Seccion 3 solo se va a mostrar t-SNE.")

try:
    from bertviz import head_view
    BERTVIZ_AVAILABLE = True
except ImportError:
    BERTVIZ_AVAILABLE = False
    print("[aviso] bertviz no disponible: en la Seccion 5 se va a usar un heatmap con matplotlib.")

RANDOM_SEED = 42  # semilla fija para que los resultados sean reproducibles entre corridas
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Imports listos.")


---
## Sección 1 — Tokenización comparada (≈20 min)

Los distintos modelos de lenguaje usan distintos algoritmos de tokenización:

- **BPE (Byte-Pair Encoding)** — usado por GPT-2: fusiona pares de caracteres/subpalabras
  frecuentes de forma iterativa.
- **WordPiece** — usado por BERT: similar a BPE, pero elige fusiones que maximizan la
  probabilidad del corpus de entrenamiento.
- Un tokenizer **multilingüe** entrenado sobre decenas de idiomas suele fragmentar menos
  el español que un tokenizer entrenado solo en inglés.

Vamos a comparar 3 tokenizers sobre el mismo texto en español e inglés, y a medir la
**fragmentación** (cuántos tokens hacen falta, en promedio, para representar una palabra).

In [ ]:
# Cargamos 3 tokenizers con estrategias distintas.
# Ojo: solo descargamos el tokenizer (vocabulario/merges), no el modelo completo.
tokenizer_configs = {
    "GPT-2 (BPE)": "gpt2",
    "BERT (WordPiece, EN)": "bert-base-uncased",
    "BERT multilingue (WordPiece)": "bert-base-multilingual-cased",
}

tokenizers = {
    name: AutoTokenizer.from_pretrained(model_id)
    for name, model_id in tokenizer_configs.items()
}

print("Tokenizers cargados:", list(tokenizers.keys()))


In [ ]:
# Mismo contenido en espanol e ingles
sample_texts = {
    "es": "La inteligencia artificial esta transformando la forma en que procesamos el lenguaje natural.",
    "en": "Artificial intelligence is transforming the way we process natural language.",
}


def tokens_per_word(text: str, tokenizer) -> tuple[int, int, float]:
    """Devuelve (num_tokens, num_words, ratio_tokens_por_palabra) para un texto y tokenizer dados."""
    num_words = len(text.split())
    # add_special_tokens=False para no contar [CLS]/[SEP]/etc. como fragmentacion "real"
    num_tokens = len(tokenizer.encode(text, add_special_tokens=False))
    ratio = num_tokens / num_words
    return num_tokens, num_words, ratio


rows = []
for tok_name, tok in tokenizers.items():
    for lang, text in sample_texts.items():
        num_tokens, num_words, ratio = tokens_per_word(text, tok)
        rows.append(
            {"tokenizer": tok_name, "idioma": lang, "tokens": num_tokens, "palabras": num_words, "ratio": ratio}
        )

df_ratios = pd.DataFrame(rows)
df_ratios


In [ ]:
# Reorganizamos la tabla: una fila por tokenizer, una columna por idioma, valores = ratio
fig, ax = plt.subplots(figsize=(8, 5))
df_pivot = df_ratios.pivot(index="tokenizer", columns="idioma", values="ratio")
df_pivot.plot(kind="bar", ax=ax)  # un grupo de barras por tokenizer (una barra por idioma)
ax.set_ylabel("Tokens por palabra (fragmentacion)")
ax.set_title("Fragmentacion por tokenizer e idioma")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1)  # referencia: 1 token = 1 palabra
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


#### 🔢 Viendo los tokens (y sus IDs)

Un tokenizer no solo parte el texto en pedazos: a cada pedazo le asigna un **ID** (un número entero que es la posición de ese token en el vocabulario). Eso es lo que el modelo realmente recibe como input — nunca ve el texto, ve una secuencia de números. Vamos a visualizar ambas cosas juntas: el token (como texto) y su ID.

In [ ]:
def visualize_tokens(text: str, tokenizer, title: str, ax=None) -> None:
    """Dibuja cada token como un recuadro de color, con el texto arriba y su ID (numero en el
    vocabulario) abajo. Asi se ve lo que el modelo realmente recibe: numeros, no texto.
    """
    ids = tokenizer.encode(text, add_special_tokens=False)  # texto -> lista de IDs numericos
    tokens = tokenizer.convert_ids_to_tokens(ids)  # los mismos IDs, mostrados como texto

    show_own_ax = ax is None
    if show_own_ax:
        # Si no nos pasaron un eje (ax), creamos una figura propia con ancho proporcional
        # a la cantidad de tokens (para que no queden amontonados).
        fig, ax = plt.subplots(figsize=(min(1.2 * len(tokens), 16), 2))

    colors = plt.cm.tab20(np.linspace(0, 1, len(tokens)))  # un color distinto por token
    x = 0.0  # posicion horizontal donde dibujar el proximo recuadro
    for token, token_id, color in zip(tokens, ids, colors):
        label = token.replace("Ġ", "␣").replace("▁", "␣")  # marca visible el espacio inicial
        width = max(0.6, 0.22 * len(label))  # recuadros mas anchos para tokens mas largos
        ax.add_patch(plt.Rectangle((x, 0), width, 1, facecolor=color, edgecolor="black"))
        ax.text(x + width / 2, 0.65, label, ha="center", va="center", fontsize=9)  # el token
        ax.text(x + width / 2, 0.25, str(token_id), ha="center", va="center", fontsize=8, color="dimgray")  # su ID
        x += width + 0.05  # avanzamos para el siguiente recuadro, con un pequeno espacio

    ax.set_xlim(0, x)
    ax.set_ylim(0, 1)
    ax.axis("off")  # sin ejes numericos, esto es un dibujo, no un grafico de datos
    ax.set_title(title, fontsize=10)
    if show_own_ax:
        plt.tight_layout()
        plt.show()


# Comparamos los 3 tokenizers sobre el mismo texto en espanol, uno debajo del otro
fig, axes = plt.subplots(len(tokenizers), 1, figsize=(14, 2 * len(tokenizers)))
for ax, (tok_name, tok) in zip(axes, tokenizers.items()):
    visualize_tokens(sample_texts["es"], tok, tok_name, ax=ax)
plt.tight_layout()
plt.show()


### Ejercicio 1

Probá con un texto propio: jerga técnica (nombres de librerías, términos de programación),
texto con errores de tipeo, o una mezcla de español e inglés. ¿Qué tokenizer fragmenta más tu
texto? ¿Por qué creés que pasa eso?

In [ ]:
# EJERCICIO: reemplaza este texto por uno propio y volve a correr la comparacion.
my_text = "Reemplazame por tu propio texto (jerga tecnica, typos, lo que quieras probar)"  # TODO

for tok_name, tok in tokenizers.items():
    num_tokens, num_words, ratio = tokens_per_word(my_text, tok)
    print(f"{tok_name:32s} -> tokens={num_tokens:3d}  palabras={num_words:3d}  ratio={ratio:.2f}")

# Bonus: mira los tokens en si (no solo el conteo) para entender COMO se parte tu texto
for tok_name, tok in tokenizers.items():
    print(f"\n{tok_name}:")
    print(" ", tok.tokenize(my_text))


#### 🤔 Para fijar ideas

1. ¿Por qué el mismo texto genera más o menos tokens según el idioma (mirá el gráfico de más arriba)? Si una API te cobra por token, ¿qué implica esto en costo y latencia?
2. Si tuvieras que tokenizar **código fuente** (Python, SQL) en vez de lenguaje natural, ¿cuál de los tres tokenizers esperás que fragmente menos? ¿Por qué?

_(Escribí tus respuestas acá)_

- 1.
- 2.

In [ ]:
# 🔎 Para explorar: probá tokenizar algo bien distinto a una oracion comun
# (una URL, un numero de telefono, un emoji, una palabra inventada) y mira que hace cada tokenizer.
texto_raro = "Reemplazame: https://ejemplo.com/api?id=123 🚀 asdkjhaslkdjh"

for tok_name, tok in tokenizers.items():
    print(f"{tok_name}:")
    print(" ", tok.tokenize(texto_raro))

# Bonus: la misma comparacion, pero viendo tokens + IDs juntos
fig, axes = plt.subplots(len(tokenizers), 1, figsize=(14, 2 * len(tokenizers)))
for ax, (tok_name, tok) in zip(axes, tokenizers.items()):
    visualize_tokens(texto_raro, tok, tok_name, ax=ax)
plt.tight_layout()
plt.show()


### Ejercicio 1b — de token a ID, en español y en inglés

Cuando tokenizás, cada pedazo de texto se convierte en un **ID**: un número entero que es la posición de ese token en el vocabulario. Esa secuencia de números es lo que la red neuronal recibe como input — nunca ve letras. Vamos a comparar cómo cambia esa secuencia de IDs entre español e inglés para la **misma frase**.

1. ¿La misma frase (en dos idiomas distintos) produce la misma cantidad de tokens? ¿Los mismos IDs? ¿Tiene sentido que sean distintos?
2. Probá con una palabra que se traduzca "distinto" entre los dos idiomas (con tilde, compuesta, jerga) y volvé a correr la celda. ¿Notás más fragmentación en un idioma que en el otro?

In [ ]:
# EJERCICIO: escribi la misma frase corta en español y su traduccion en ingles.
frase_es = "el perro corre en el parque"  # TODO: cambiala por la tuya
frase_en = "the dog runs in the park"  # TODO: traducila vos mismo

for lang_name, frase in [("Español", frase_es), ("Ingles", frase_en)]:
    print(f'\n=== {lang_name}: "{frase}" ===')
    for tok_name, tok in tokenizers.items():
        ids = tok.encode(frase, add_special_tokens=False)
        toks = tok.convert_ids_to_tokens(ids)
        print(f"  {tok_name}:")
        print(f"    tokens: {toks}")
        print(f"    ids:    {ids}")

# Visualizacion lado a lado (token + ID) para comparar de un vistazo
fig, axes = plt.subplots(2 * len(tokenizers), 1, figsize=(14, 2 * len(tokenizers) * 2))
row = 0
for lang_name, frase in [("Español", frase_es), ("Ingles", frase_en)]:
    for tok_name, tok in tokenizers.items():
        visualize_tokens(frase, tok, f"{tok_name} — {lang_name}", ax=axes[row])
        row += 1
plt.tight_layout()
plt.show()


---
## Sección 2 — Embeddings clásicos con Word2Vec (≈20 min)

Word2Vec aprende un vector por palabra a partir de su contexto (skip-gram o CBOW): palabras
que aparecen en contextos parecidos terminan con vectores parecidos.

Primero entrenamos un modelo chico sobre un corpus de juguete para ver el mecanismo de
entrenamiento en acción. **Ojo:** un corpus de un puñado de oraciones no alcanza para obtener
vectores útiles — es solo para ver el proceso. Para explorar similitudes y analogías de verdad,
después cargamos vectores **pre-entrenados** (GloVe, en inglés, vía `gensim.downloader`).

In [ ]:
# Corpus de juguete: unas pocas oraciones sobre un tema comun (realeza / familia / mascotas).
# Sirve para ver COMO se entrena Word2Vec, no para obtener embeddings de calidad.
toy_corpus = [
    "el rey gobierna el reino con sabiduria".split(),  # Word2Vec espera listas de palabras, no strings
    "la reina gobierna el reino con justicia".split(),
    "el hombre camina por la calle del pueblo".split(),
    "la mujer camina por la calle del pueblo".split(),
    "el principe hereda el trono del rey".split(),
    "la princesa hereda el trono de la reina".split(),
    "el rey y la reina viven en el castillo".split(),
    "el hombre y la mujer viven en la ciudad".split(),
    "un perro y un gato juegan en el jardin".split(),
    "el gato duerme en el sillon todo el dia".split(),
    "el perro corre detras de la pelota".split(),
    "el gato caza al raton en la noche".split(),
]

toy_model = Word2Vec(
    sentences=toy_corpus,
    vector_size=50,  # dimension de cada vector de palabra (GloVe mas adelante usa 100)
    window=3,  # cuantas palabras a cada lado se consideran "contexto"
    min_count=1,  # incluir palabras aunque aparezcan una sola vez (corpus chico)
    sg=1,  # skip-gram (mejor que CBOW para corpus chicos)
    seed=RANDOM_SEED,
    epochs=200,  # muchas epocas para compensar que el corpus es diminuto
)

print("Vocabulario del modelo de juguete:", list(toy_model.wv.index_to_key))
print("\nPalabras mas 'similares' a 'rey' segun el modelo de juguete (poco confiable, corpus chico):")
for word, score in toy_model.wv.most_similar("rey", topn=5):  # most_similar = similitud coseno entre vectores
    print(f"  {word:12s} {score:.3f}")


Como se ve arriba, con tan pocas oraciones las similitudes no son muy confiables. Para
explorar similitudes y analogías de verdad usamos vectores pre-entrenados sobre un corpus real
(Wikipedia + noticias). `gensim-data` ofrece estos sets mayormente en **inglés**, así que de
acá en adelante trabajamos en inglés.

In [ ]:
# Vectores pre-entrenados: GloVe, 100 dimensiones, entrenado sobre Wikipedia + Gigaword.
# La descarga pesa ~130MB y puede tardar uno o dos minutos la primera vez.
glove_vectors = gensim_api.load("glove-wiki-gigaword-100")
print("Vectores cargados. Tamano del vocabulario:", len(glove_vectors.index_to_key))


In [ ]:
# Para cada palabra, pedimos las 5 mas parecidas segun el vector GloVe (similitud coseno)
for word in ["king", "computer", "spain"]:
    print(f"\nPalabras mas similares a '{word}':")
    for similar_word, score in glove_vectors.most_similar(word, topn=5):
        print(f"  {similar_word:15s} {score:.3f}")  # score cercano a 1 = muy similar


In [ ]:
# Analogia clasica: rey - hombre + mujer = ?
result = glove_vectors.most_similar(positive=["king", "woman"], negative=["man"], topn=5)
print("king - man + woman = ?")
for word, score in result:
    print(f"  {word:15s} {score:.3f}")


### Ejercicio 2

Probá otras analogías con tus propias palabras (en inglés, porque los vectores pre-entrenados
que cargamos son en inglés). Intentá **romper** la analogía clásica con palabras donde el
resultado no sea el esperado — ¿con qué tipo de palabras falla?

In [ ]:
# EJERCICIO: cambia estas listas y volve a correr. Ejemplo que deberia funcionar:
#   positive=["paris", "italy"], negative=["france"] -> se espera algo cercano a "rome"
positive_words = ["paris", "italy"]  # TODO: proba con tus propias palabras
negative_words = ["france"]  # TODO

result = glove_vectors.most_similar(positive=positive_words, negative=negative_words, topn=5)
for word, score in result:
    print(f"  {word:15s} {score:.3f}")

# Ahora proba con palabras donde sospeches que la analogia se "rompe"
# (por ejemplo, profesiones, o pares donde el estereotipo de genero no aplica bien)


#### 🤔 Para fijar ideas

1. ¿Por qué el corpus de juguete (unas pocas oraciones) no alcanzó para tener buenas similitudes, mientras que GloVe sí funciona bien? ¿Qué relación hay entre tamaño del corpus y calidad de los embeddings?
2. Buscá una analogía que **no** haya funcionado bien en tu prueba de arriba. ¿Qué te dice eso sobre los sesgos o límites del corpus con el que se entrenó GloVe?

_(Escribí tus respuestas acá)_

- 1.
- 2.

In [ ]:
# 🔎 Para explorar: proba most_similar con una palabra tuya, y una analogia inventada por vos.
mi_palabra = "computer"  # cambiala
print(glove_vectors.most_similar(mi_palabra, topn=5))

# Analogia libre: positive=[...] - negative=[...]
# print(glove_vectors.most_similar(positive=[...], negative=[...], topn=5))


#### 📐 Visualizando la analogía vectorialmente

La ecuación `king - man + woman ≈ queen` dice que el vector que va de *man* a *woman* es (aproximadamente) el mismo que el que va de *king* a *queen*: ambos capturan la misma "dirección de género" dentro del espacio de embeddings. Vamos a proyectar estas 4 palabras a 2D con **PCA** (a diferencia de t-SNE, PCA preserva mejor las direcciones relativas entre puntos, ideal para ver si dos flechas son paralelas) y dibujar esa relación como flechas.

In [ ]:
def plot_analogy_2d(word_pairs: list[tuple[str, str]], title: str) -> None:
    """Proyecta a 2D con PCA los pares de palabras y dibuja una flecha del primero al segundo
    de cada par, para ver si la relacion vectorial (ej. man->woman) es paralela a otra (ej.
    king->queen).
    """
    all_words = [w for pair in word_pairs for w in pair]  # aplanamos los pares en una sola lista
    vectors = np.array([glove_vectors[w] for w in all_words])  # sus vectores GloVe (100-dim)

    # PCA reduce las 100 dimensiones a 2, tratando de preservar las distancias relativas
    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    coords = pca.fit_transform(vectors)

    fig, ax = plt.subplots(figsize=(6, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(word_pairs)))  # un color por par

    for i, (word_a, word_b) in enumerate(word_pairs):
        idx_a, idx_b = 2 * i, 2 * i + 1  # cada par ocupa 2 posiciones consecutivas en "coords"
        ax.annotate(  # dibuja la flecha de word_a a word_b
            "", xy=coords[idx_b], xytext=coords[idx_a],
            arrowprops=dict(arrowstyle="->", color=colors[i], lw=2),
        )
        for idx, word in [(idx_a, word_a), (idx_b, word_b)]:  # dibuja los dos puntos + su etiqueta
            ax.scatter(*coords[idx], color=colors[i], s=60)
            ax.annotate(word, coords[idx], fontsize=11, xytext=(5, 5), textcoords="offset points")

    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# Si las dos flechas apuntan en (casi) la misma direccion, la relacion "de genero"
# (man->woman) es la misma que la relacion "de titulo" (king->queen).
plot_analogy_2d([("man", "woman"), ("king", "queen")], "man→woman vs king→queen: ¿son paralelas?")


### Ejercicio 2b — probá tu propia analogía

Elegí tu propio par de pares de palabras (por ejemplo `("paris", "france")` vs `("rome", "italy")`, o `("actor", "actress")` vs `("king", "queen")`) y llamá a `plot_analogy_2d` con ellos.

1. ¿Las flechas te parecen paralelas (misma dirección y longitud aproximada)?
2. Si no lo son, ¿qué te dice eso sobre la relación semántica que elegiste?

_(Escribí tus respuestas acá)_

- 1.
- 2.

In [ ]:
# EJERCICIO: cambia estos pares por los tuyos y volve a correr.
plot_analogy_2d(
    [("man", "woman"), ("king", "queen")],  # TODO: reemplaza por tus propios pares de palabras
    "Mi propia comparacion de analogias",
)


---
## Sección 3 — Visualización de embeddings: t-SNE vs UMAP (≈20-25 min)

Los embeddings de GloVe tienen 100 dimensiones — imposibles de graficar directamente. Usamos dos
técnicas de reducción de dimensionalidad para llevarlos a 2D y ver si palabras semánticamente
parecidas (misma categoría) quedan agrupadas:

- **t-SNE**: preserva bien las relaciones de vecindad local, pero es sensible a hiperparámetros
  (`perplexity`) y no preserva bien distancias globales.
- **UMAP**: suele ser más rápido y preservar mejor tanto estructura local como global.

In [ ]:
# Grupos de palabras por categoria semantica (en ingles, para que coincidan con el vocabulario de GloVe)
word_categories = {
    "animals": ["dog", "cat", "lion", "tiger", "elephant", "mouse", "horse", "bird"],
    "countries": ["spain", "france", "germany", "italy", "brazil", "argentina", "japan", "china"],
    "colors": ["red", "blue", "green", "yellow", "black", "white", "purple", "orange"],
    "food": ["bread", "cheese", "wine", "rice", "pizza", "sushi", "chocolate", "coffee"],
}


def build_word_vectors(categories_dict: dict) -> tuple[list[str], list[str], np.ndarray]:
    """Aplana el diccionario categoria -> palabras en listas paralelas + matriz de vectores."""
    words_, categories_ = [], []
    for category, word_list in categories_dict.items():
        for w in word_list:
            if w in glove_vectors.key_to_index:  # nos aseguramos de que el vector exista
                words_.append(w)
                categories_.append(category)  # misma posicion que words_, para saber la categoria de cada palabra
    vectors_ = np.array([glove_vectors[w] for w in words_])  # matriz (num_palabras, 100)
    return words_, categories_, vectors_


words, categories, vectors = build_word_vectors(word_categories)
print(f"{len(words)} palabras con vector disponible.")


In [ ]:
def plot_2d_embeddings(coords: np.ndarray, words: list[str], categories: list[str], title: str) -> None:
    """Grafica puntos 2D coloreados por categoria, con el texto de la palabra al lado de cada punto."""
    fig, ax = plt.subplots(figsize=(9, 7))
    unique_categories = sorted(set(categories))
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_categories)))  # un color por categoria
    color_map = dict(zip(unique_categories, colors))

    for category in unique_categories:
        idx = [i for i, c in enumerate(categories) if c == category]  # indices de esta categoria
        ax.scatter(coords[idx, 0], coords[idx, 1], label=category, color=color_map[category], s=60)

    for i, word in enumerate(words):  # etiqueta con el texto de la palabra al lado de cada punto
        ax.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=8, alpha=0.8)

    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


# t-SNE reduce los 100-dim de GloVe a 2D preservando sobre todo la vecindad local
tsne = TSNE(n_components=2, perplexity=15, random_state=RANDOM_SEED, init="pca")
vectors_tsne = tsne.fit_transform(vectors)
plot_2d_embeddings(vectors_tsne, words, categories, "t-SNE de embeddings GloVe por categoria")


In [ ]:
# UMAP es una alternativa a t-SNE: suele preservar mejor tanto la estructura local como la global
if UMAP_AVAILABLE:
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED, n_neighbors=10, min_dist=0.3)
    vectors_umap = reducer.fit_transform(vectors)
    plot_2d_embeddings(vectors_umap, words, categories, "UMAP de embeddings GloVe por categoria")
else:
    print("umap-learn no esta disponible en este entorno.")
    print("Instalalo con `!pip install umap-learn` y reinicia el runtime para reintentar.")


### Ejercicio 3

Agregá una categoría nueva (por ejemplo `sports`, `emotions` o `professions`) con 6-8 palabras
en inglés, y volvé a correr las celdas de arriba (desde `build_word_vectors` en adelante).
¿El grupo nuevo queda separado del resto, o se mezcla con alguna categoría existente? ¿Cambia
la respuesta entre t-SNE y UMAP?

In [ ]:
# EJERCICIO: agrega tu propia categoria y palabras aca.
# TODO 1: elegi un nombre de categoria y 4-8 palabras en ingles (para que existan en GloVe)
word_categories["___"] = ["___", "___", "___", "___"]

# TODO 2: volve a construir los vectores (podes reusar build_word_vectors de la celda de arriba)
# words, categories, vectors = build_word_vectors(word_categories)

# TODO 3: volve a graficar con t-SNE o UMAP (podes copiar/pegar el codigo de las celdas de arriba)
# tsne_ex = TSNE(n_components=2, perplexity=15, random_state=RANDOM_SEED, init="pca")
# vectors_tsne_ex = tsne_ex.fit_transform(vectors)
# plot_2d_embeddings(vectors_tsne_ex, words, categories, "t-SNE con la categoria nueva")


#### 🤔 Para fijar ideas

1. ¿t-SNE y UMAP agruparon las categorías de la misma forma? ¿Cuál te resultó visualmente más clara?
2. Si corrieras t-SNE dos veces con distinta semilla aleatoria, ¿esperás exactamente el mismo gráfico? ¿Qué implica esto a la hora de sacar conclusiones de una única visualización?

_(Escribí tus respuestas acá)_

- 1.
- 2.

In [ ]:
# 🔎 Para explorar: cambia el hiperparametro `perplexity` de TSNE (proba 5 y 50) y volve a
# graficar. Si tenes UMAP disponible, proba tambien distintos valores de n_neighbors.
tsne_explore = TSNE(n_components=2, perplexity=5, random_state=RANDOM_SEED, init="pca")
vectors_tsne_explore = tsne_explore.fit_transform(vectors)
plot_2d_embeddings(vectors_tsne_explore, words, categories, "t-SNE (perplexity=5)")


---
## Sección 4 — Self-Attention y Multi-Head Attention desde cero (≈25 min)

La atención de producto punto escalado se define como:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Donde $Q$ (queries), $K$ (keys) y $V$ (values) son proyecciones lineales del mismo input (en
self-attention). Dividir por $\sqrt{d_k}$ evita que los productos punto crezcan demasiado y
saturen el softmax.

**Multi-head attention** corre varias "cabezas" de atención en paralelo, cada una mirando una
sub-porción de las dimensiones, y después concatena los resultados. Esto le permite al modelo
capturar distintos tipos de relaciones (sintácticas, semánticas, posicionales) al mismo tiempo.

Vamos a implementar ambas piezas **sin usar `nn.MultiheadAttention`**, para entender el mecanismo
paso a paso.

In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask: Optional[torch.Tensor] = None
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Calcula la atencion escalada de producto punto.

    Q, K, V: tensores de shape (..., seq_len, d_k)
    mask: opcional, shape broadcastable a (..., seq_len_q, seq_len_k). Las posiciones con
          mask == 0 se anulan (se les asigna -inf antes del softmax), util para atencion causal.

    Devuelve: (output, attention_weights)
    """
    d_k = Q.size(-1)
    # Q @ K^T: para cada posicion, cuanto "coincide" su query con la key de cada otra posicion.
    # Se divide por sqrt(d_k) para evitar que estos productos crezcan demasiado con d_k grande
    # (lo que empujaria al softmax a valores extremos y gradientes muy chicos).
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)  # (..., seq_len_q, seq_len_k)

    if mask is not None:
        # -inf antes del softmax = 0 de probabilidad despues del softmax (esa posicion "no se ve")
        scores = scores.masked_fill(mask == 0, float("-inf"))

    attention_weights = F.softmax(scores, dim=-1)  # convierte los scores en "pesos" que suman 1 por fila
    output = torch.matmul(attention_weights, V)  # combinacion ponderada de los valores V
    return output, attention_weights


In [ ]:
# Probamos la funcion standalone con un tensor chico y verificamos shapes
batch_size, seq_len, d_k = 2, 4, 8
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, attn = scaled_dot_product_attention(Q, K, V)
print("Output shape:", output.shape)            # (batch, seq_len, d_k)
print("Attention weights shape:", attn.shape)    # (batch, seq_len, seq_len)
print("Suma de pesos de atencion por fila (debe ser ~1 por el softmax):", attn[0, 0].sum().item())


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension de cada cabeza individual

        # Proyecciones lineales: se aplican sobre todo d_model y despues se dividen en cabezas
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # proyeccion de salida, combina las cabezas

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
        # Partimos el vector de cada posicion en "num_heads" pedacitos de tamano d_k,
        # para que cada cabeza aprenda a prestar atencion a cosas distintas.
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        # (batch, num_heads, seq_len, d_k) -> (batch, seq_len, d_model)
        # Operacion inversa a split_heads: volvemos a pegar las cabezas en un solo vector.
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.d_model)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> tuple[torch.Tensor, torch.Tensor]:
        # Proyectamos x en Q, K y V, y partimos cada uno en "num_heads" cabezas
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # Cada cabeza calcula su propia atencion en paralelo (misma funcion de la celda anterior)
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # Volvemos a juntar las cabezas y proyectamos a la salida final
        output = self.combine_heads(attn_output)
        output = self.W_o(output)
        return output, attn_weights


In [ ]:
# Probamos la clase completa con un tensor de ejemplo
d_model, num_heads, seq_len, batch_size = 16, 4, 5, 2
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

x = torch.randn(batch_size, seq_len, d_model)
output, attn_weights = mha(x)

print("Input shape: ", x.shape)                        # (2, 5, 16)
print("Output shape:", output.shape)                    # (2, 5, 16) -> mismo shape que el input
print("Attention weights shape:", attn_weights.shape)    # (2, 4, 5, 5) -> (batch, heads, seq, seq)


### Ejercicio 4

Probá con distinta cantidad de cabezas (`num_heads`) y confirmá que:

1. `output.shape` se mantiene igual a `(batch_size, seq_len, d_model)` sin importar `num_heads`.
2. `attn_weights.shape` cambia su segunda dimensión según `num_heads`.
3. `d_model` debe ser divisible por `num_heads` — probá un valor que **no** lo sea y mirá el error.

In [ ]:
# EJERCICIO: proba distintos valores de num_heads (que dividan a d_model=16: 1, 2, 4, 8, 16)
for num_heads_try in [1, 2, 8]:  # TODO: agrega o cambia valores
    mha_test = MultiHeadAttention(d_model=d_model, num_heads=num_heads_try)
    out, attn = mha_test(x)
    print(f"num_heads={num_heads_try:2d} -> output={tuple(out.shape)}  attn_weights={tuple(attn.shape)}")

# Descomenta la siguiente linea para ver el AssertionError cuando d_model no es divisible por num_heads:
# mha_bad = MultiHeadAttention(d_model=d_model, num_heads=5)


#### 🤔 Para fijar ideas

1. ¿Por qué dividimos por √d_k en la fórmula de atención? ¿Qué pasaría con los productos punto si `d_k` fuera muy grande y no dividiéramos?
2. ¿Qué ventaja práctica tiene usar varias cabezas chicas en paralelo en vez de una sola cabeza grande?

_(Escribí tus respuestas acá)_

- 1.
- 2.

In [ ]:
# 🔎 Para explorar: agrega una mascara causal (triangular inferior) a la atencion, para que
# cada posicion solo pueda "ver" las anteriores (como en un modelo generador tipo GPT).
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)  # (1, seq_len, seq_len)
out_causal, attn_causal = mha(x, mask=causal_mask)
print("Pesos de atencion (cabeza 0) con mascara causal:")
print(attn_causal[0, 0].round(decimals=2))  # deberia ser triangular: ceros por encima de la diagonal


---
## Sección 5 — Atención en un modelo real: DistilBERT vs GPT-2 (≈25 min)

Ahora que implementamos atención desde cero, vamos a mirar los pesos de atención **reales** de
un modelo pre-entrenado. Comparamos:

- **DistilBERT** (`distilbert-base-uncased`): modelo *encoder-only*, atención **bidireccional**
  (cada token puede atender a cualquier otro token de la oración, incluidos los que vienen después).
- **GPT-2**: modelo *decoder-only*, atención **causal** (cada token solo puede atender a sí mismo
  y a los tokens anteriores — nunca a los que vienen después).

In [ ]:
# Cargamos DistilBERT (encoder-only) con output_attentions=True para poder inspeccionar
# los pesos de atencion de cada capa/cabeza, no solo la prediccion final.
bert_model_name = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name, output_attentions=True)
bert_model.eval()  # modo evaluacion: desactiva dropout, no vamos a entrenar nada

sentence = "The animal didn't cross the street because it was too tired"

inputs = bert_tokenizer(sentence, return_tensors="pt")  # texto -> tensores de IDs listos para el modelo
with torch.no_grad():  # no necesitamos gradientes, solo inferencia (ahorra memoria y tiempo)
    outputs = bert_model(**inputs)

# outputs.attentions: tupla con un tensor por capa, shape (batch, num_heads, seq_len, seq_len)
bert_attentions = outputs.attentions
tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"Numero de capas: {len(bert_attentions)}")
print(f"Shape de la atencion en una capa: {bert_attentions[0].shape}")
print(f"Tokens: {tokens}")


In [ ]:
# Visualizacion interactiva con bertviz si esta disponible; si no, heatmap manual con matplotlib.
layer, head = 5, 0  # proba cambiar estos valores en el Ejercicio 5

if BERTVIZ_AVAILABLE:
    head_view(bert_attentions, tokens)
else:
    attn_matrix = bert_attentions[layer][0, head].numpy()

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(attn_matrix, cmap="viridis")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticklabels(tokens)
    ax.set_title(f"DistilBERT (encoder) - atencion capa {layer}, cabeza {head}")
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()


In [ ]:
# Ahora cargamos GPT-2 (decoder-only) y repetimos con la misma oracion, para comparar
gpt2_model_name = "gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(gpt2_model_name)
gpt2_model = AutoModelForCausalLM.from_pretrained(gpt2_model_name, output_attentions=True)
gpt2_model.eval()

gpt2_inputs = gpt2_tokenizer(sentence, return_tensors="pt")  # misma oracion que con DistilBERT
with torch.no_grad():
    gpt2_outputs = gpt2_model(**gpt2_inputs)

gpt2_attentions = gpt2_outputs.attentions
gpt2_tokens = gpt2_tokenizer.convert_ids_to_tokens(gpt2_inputs["input_ids"][0])

print(f"Numero de capas GPT-2: {len(gpt2_attentions)}")
print(f"Shape de la atencion en una capa: {gpt2_attentions[0].shape}")


In [ ]:
# Heatmap comparativo lado a lado: DistilBERT (bidireccional) vs GPT-2 (causal)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Panel izquierdo: DistilBERT ---
bert_matrix = bert_attentions[layer][0, head].numpy()  # [0]=primer (unico) elemento del batch
axes[0].imshow(bert_matrix, cmap="viridis")
axes[0].set_xticks(range(len(tokens)))
axes[0].set_yticks(range(len(tokens)))
axes[0].set_xticklabels(tokens, rotation=90)
axes[0].set_yticklabels(tokens)
axes[0].set_title(f"DistilBERT (encoder) - capa {layer}, cabeza {head}")

# --- Panel derecho: GPT-2 (puede tener menos capas que DistilBERT, por eso el min()) ---
gpt2_layer = min(layer, len(gpt2_attentions) - 1)
gpt2_matrix = gpt2_attentions[gpt2_layer][0, head].numpy()
axes[1].imshow(gpt2_matrix, cmap="viridis")
axes[1].set_xticks(range(len(gpt2_tokens)))
axes[1].set_yticks(range(len(gpt2_tokens)))
axes[1].set_xticklabels(gpt2_tokens, rotation=90)
axes[1].set_yticklabels(gpt2_tokens)
axes[1].set_title(f"GPT-2 (decoder) - capa {gpt2_layer}, cabeza {head}")

plt.tight_layout()
plt.show()


### Ejercicio 5

Compará las dos matrices de atención de arriba y respondé acá mismo (doble click para editar
esta celda):

1. ¿Qué forma tiene la matriz de atención de GPT-2 que **no** tiene la de DistilBERT? ¿Por qué
   creés que pasa eso? (pensá en cómo se entrena cada modelo — predicción de la próxima palabra
   vs. modelado bidireccional).
2. Probá cambiar `layer` y `head` en la celda del heatmap comparativo y volvé a correrla.
   ¿Encontrás alguna cabeza que preste atención principalmente al token anterior? ¿O al primer
   token de la oración?
3. En la oración de ejemplo, `it` es ambiguo (podría referirse a `animal` o a `street`). ¿Alguna
   cabeza de atención de DistilBERT parece "resolver" esa ambigüedad prestando atención de `it`
   hacia `animal`?

_(Escribí tus respuestas acá)_

- 1.
- 2.
- 3.


In [ ]:
# EJERCICIO (opcional, codigo): proba distintas combinaciones de layer/head.
# Cambia estos valores y volve a correr la celda del heatmap comparativo de arriba.
layer, head = 3, 2  # TODO: proba otras combinaciones (layer va de 0 a 5, head de 0 a 11)


#### 🤔 Para fijar ideas: síntesis de toda la práctica

1. Recorriste tokenización → embeddings → visualización → atención. ¿En qué etapa se "pierde" menos información sobre el significado de una palabra: al tokenizarla o al embeberla?
2. Si tuvieras que explicarle a alguien sin background técnico qué hace "la atención" en una sola frase, ¿qué dirías?

_(Escribí tus respuestas acá)_

- 1.
- 2.

In [ ]:
# 🔎 Para explorar: repeti la comparacion encoder vs decoder con una oracion propia,
# idealmente una con ambiguedad (un pronombre que podria referirse a mas de una cosa).
mi_oracion = "Reemplazame por tu propia oracion ambigua"
# Sugerencia: copia y adapta el codigo de las celdas de arriba (bert_tokenizer/gpt2_tokenizer)
# usando `mi_oracion` en vez de `sentence`, y volve a graficar el heatmap comparativo.


---
## Cierre

En esta práctica recorrimos toda la cadena de procesamiento de un LLM moderno: **tokenización**
(cómo se parte el texto), **embeddings** (cómo se representa el significado como vectores),
**visualización** (cómo "ver" esos vectores en 2D), y **atención** (el mecanismo central que le
permite a un Transformer relacionar palabras entre sí, implementado primero desde cero y después
observado en modelos reales).

**Para seguir explorando:**
- Repetir la Sección 5 con una oración ambigua propia.
- Probar tokenizers/modelos de otras familias (T5, Llama, Mistral) y comparar.
- Entrenar el `MultiHeadAttention` de la Sección 4 como parte de un Transformer completo
  (agregando feed-forward, layer norm y conexiones residuales).